# Generic dataset preparation

Notebook tổng quát để tạo dense subset (tùy chọn), lọc review bằng LLM, tạo split 8/1/1 và đánh giá semantic retention cho một hoặc nhiều dataset. Toàn bộ logic nằm trong `preprocessing_reviews.py`; notebook chỉ cấu hình và gọi CLI.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'preprocessing_reviews.py').is_file():
    if REPO_ROOT.parent == REPO_ROOT:
        raise FileNotFoundError('Không tìm thấy repository PreBERT')
    REPO_ROOT = REPO_ROOT.parent

def run(*arguments):
    command = [sys.executable, *map(str, arguments)]
    print('$', ' '.join(command))
    subprocess.run(command, cwd=REPO_ROOT, check=True)

## Configuration

Thêm bao nhiêu dataset tùy ý. `source_url` và `raw_cache` chỉ cần khi `build_dense=True`. `domain='auto'` tự suy luận domain; dùng `general` cho domain mới.

In [ ]:
DATASETS = [
    {
        'name': 'dataset_name',
        'input': REPO_ROOT / 'data/source_dataset.json',
        'processed': REPO_ROOT / 'data/source_dataset_llama_filtered.json',
        'build_dense': False,
        'source_url': None,
        'raw_cache': None,
        'target_size': 10_000,
        'k_core': 5,
        'domain': 'auto',
    },
]

RUN_BUILD = False
RUN_PREPROCESS = False
RUN_SPLIT = False
RUN_SEMANTIC = False
OVERWRITE = False
MODEL = 'meta-llama/Llama-3.2-3B-Instruct'
DEVICE = 'auto'
SEED = 42

In [ ]:
for config in DATASETS:
    if RUN_BUILD and config['build_dense']:
        args = [
            'preprocessing_reviews.py', 'build-dataset',
            '--output', config['input'],
            '--target-size', config['target_size'],
            '--k-core', config['k_core'],
            '--seed', SEED,
        ]
        if config.get('source_url'):
            args += ['--source-url', config['source_url']]
        if config.get('raw_cache'):
            args += ['--raw-cache', config['raw_cache']]
        if OVERWRITE:
            args.append('--overwrite')
        run(*args)

    if RUN_PREPROCESS:
        args = [
            'preprocessing_reviews.py', 'preprocess', config['input'],
            '--output', config['processed'], '--model', MODEL,
            '--domain', config['domain'], '--device', DEVICE,
        ]
        if OVERWRITE:
            args.append('--overwrite')
        run(*args)

    if RUN_SPLIT:
        args = [
            'preprocessing_reviews.py', 'split', config['processed'],
            '--source', config['input'], '--seed', SEED,
        ]
        if OVERWRITE:
            args.append('--overwrite')
        run(*args)

In [ ]:
if RUN_SEMANTIC:
    for config in DATASETS:
        run(
            '-m', 'experiments', 'semantic', config['processed'],
            '--output-dir', REPO_ROOT / 'experiments/semantic_outputs' / config['name'],
            '--device', DEVICE,
        )